# Model 12: Syllable-to-Syllable Decipherment

## Hypothesis
If Voynich glyphs are sub-syllabic components (onset/nucleus/coda), then the correct
decipherment unit is the **Voynich syllable** (2-3 glyph group), not the individual glyph.

This model maps Voynich syllable types → Hebrew syllable types, using:
- TP-P20 segmentation for both Voynich and Hebrew
- LSTM character-level LM for scoring (decode syllables back to characters)
- Same 5-test scorecard as all other models

Key advantage: each mapping unit already has internal positional coherence.

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────
import re, math, random, json, warnings, time, os
from collections import Counter, defaultdict
from itertools import combinations
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.spatial.distance import jensenshannon
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# ── Cell 2: Parser + TP-P20 Segmentation ─────────────────────────────

def parsear_sta_completo(filepath):
    tokens = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for linea in f:
            linea = linea.strip()
            if not linea or linea.startswith('#'):
                continue
            if re.match(r'^<[^>]+>\s*<!', linea):
                continue
            match = re.match(r'^<[^>]+>\s+(.*)', linea)
            if not match:
                continue
            texto = match.group(1).strip()
            texto = re.sub(r'\{[^}]*\}', '', texto)
            texto = re.sub(r'<[^>]*>', '', texto)
            texto = re.sub(r'-', '.', texto)
            texto = texto.strip()
            if not texto:
                continue
            for palabra_raw in texto.split('.'):
                palabra_raw = palabra_raw.strip()
                if not palabra_raw:
                    continue
                glifos = re.findall(r'[A-Z][0-9a-z]', palabra_raw)
                if glifos:
                    tokens.extend(glifos)
                    tokens.append('.')
    return tokens


def tokens_a_palabras(tokens):
    palabras, actual = [], []
    for t in tokens:
        if t == '.':
            if actual:
                palabras.append(actual)
                actual = []
        else:
            actual.append(t)
    if actual:
        palabras.append(actual)
    return palabras


def construir_matriz_transicion_intrapalabra(palabras):
    all_glyphs = sorted({g for p in palabras for g in p})
    index = {g: i for i, g in enumerate(all_glyphs)}
    n = len(all_glyphs)
    M_raw = np.zeros((n, n))
    for palabra in palabras:
        for i in range(len(palabra) - 1):
            r, c = index[palabra[i]], index[palabra[i + 1]]
            M_raw[r, c] += 1
    row_sums = M_raw.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    M_prob = M_raw / row_sums
    return M_raw, M_prob, all_glyphs, index


def segmentar_tp(palabras, M_prob, index, threshold_percentile=20):
    """Segment words into syllables using transition probability minima."""
    all_tp = []
    for palabra in palabras:
        for i in range(len(palabra) - 1):
            g1, g2 = palabra[i], palabra[i + 1]
            if g1 in index and g2 in index:
                all_tp.append(M_prob[index[g1], index[g2]])
    threshold = np.percentile(all_tp, threshold_percentile)
    
    words_as_syls = []
    for palabra in palabras:
        if not palabra:
            continue
        syl = [palabra[0]]
        for i in range(len(palabra) - 1):
            g1, g2 = palabra[i], palabra[i + 1]
            p = M_prob[index[g1], index[g2]] if (g1 in index and g2 in index) else 0.0
            if p < threshold:
                words_as_syls[-1] if words_as_syls else None  # safety
                syl = [g2]
            else:
                syl.append(g2)
            # Append completed syllable when break found
            if p < threshold:
                pass  # already started new syl
        # Collect syllables for this word
        pass
    
    # Cleaner implementation
    words_as_syls = []
    for palabra in palabras:
        if not palabra:
            continue
        syls = []
        current_syl = [palabra[0]]
        for i in range(len(palabra) - 1):
            g1, g2 = palabra[i], palabra[i + 1]
            p = M_prob[index[g1], index[g2]] if (g1 in index and g2 in index) else 0.0
            if p < threshold:
                syls.append(tuple(current_syl))
                current_syl = [g2]
            else:
                current_syl.append(g2)
        syls.append(tuple(current_syl))
        words_as_syls.append(syls)
    return words_as_syls, threshold


# ── Load Voynich ──
STA_PATH = '../../corpus/voynich_sta.txt'
tokens_v = parsear_sta_completo(STA_PATH)
palabras = tokens_a_palabras(tokens_v)

# ── R1 capacity control: glyphs permuted across the corpus (lengths kept) ──
import numpy as _np
_r = _np.random.default_rng(2031)
_toks = [g for w in palabras for g in w]
_r.shuffle(_toks)
_out, _i = [], 0
for _w in palabras:
    _out.append(type(_w)(_toks[_i:_i + len(_w)]) if not isinstance(_w, str) else ''.join(_toks[_i:_i + len(_w)]))
    _i += len(_w)
palabras = _out

VOCAB_GLYPH = sorted({g for p in palabras for g in p})
print(f'Voynich: {len(palabras):,} words, {sum(len(p) for p in palabras):,} glyphs, {len(VOCAB_GLYPH)} types')

# ── Segment Voynich ──
M_raw_v, M_prob_v, glyphs_v, idx_v = construir_matriz_transicion_intrapalabra(palabras)
words_syl_v, thresh_v = segmentar_tp(palabras, M_prob_v, idx_v, 20)

# Statistics
all_syl_v = [s for w in words_syl_v for s in w]
syl_freq_v = Counter(all_syl_v)
VOCAB_SYL_V = sorted(syl_freq_v.keys(), key=lambda s: syl_freq_v[s], reverse=True)
VOCAB_SYL_V_FUNC = [s for s in VOCAB_SYL_V if syl_freq_v[s] >= 5]
syl_per_word = [len(w) for w in words_syl_v]

print(f'Voynich syllables: {len(VOCAB_SYL_V)} types, {len(VOCAB_SYL_V_FUNC)} functional (freq>=5)')
print(f'Mean syllables/word: {np.mean(syl_per_word):.2f}')
print(f'Mean glyphs/syllable: {np.mean([len(s) for s in all_syl_v]):.2f}')
print(f'Threshold: {thresh_v:.4f}')

In [ ]:
# ── Cell 3: Hebrew corpus + syllabic segmentation ────────────────────
import requests

def descargar_hebreo():
    libros = ['Genesis', 'Exodus', 'Leviticus', 'Numbers', 'Deuteronomy']
    base_url = ('https://raw.githubusercontent.com/Sefaria/Sefaria-Export/'
                'master/json/Tanakh/Torah/{}/Hebrew/Tanach%20with%20Text%20Only.json')
    todas_palabras = []
    for libro in libros:
        url = base_url.format(libro)
        print(f'  Descargando {libro}...')
        data = json.load(open(f'../../corpus/sefaria/torah_hebrew/{libro}.json', encoding='utf-8'))
        for capitulo in data['text']:
            for versiculo in capitulo:
                texto = re.sub(r'[\u0591-\u05C7]', '', versiculo)
                pals = re.findall(r'[\u05D0-\u05EA]+', texto)
                for p in pals:
                    chars = list(p)
                    if 2 <= len(chars) <= 15:
                        todas_palabras.append(chars)
    print(f'  Hebrew total: {len(todas_palabras):,} words')
    return todas_palabras

print('Downloading Hebrew Bible (Torah)...')
hebrew_words_full = descargar_hebreo()

# Subsample to match Voynich size
rng_sub = np.random.default_rng(42)
if len(hebrew_words_full) > len(palabras):
    idx_sub = rng_sub.choice(len(hebrew_words_full), size=len(palabras), replace=False)
    hebrew_words_chars = [hebrew_words_full[i] for i in sorted(idx_sub)]
else:
    hebrew_words_chars = hebrew_words_full
target_chars_heb = sorted({c for w in hebrew_words_chars for c in w})
print(f'Hebrew subsample: {len(hebrew_words_chars):,} words, {len(target_chars_heb)} chars')

# ── Segment Hebrew with TP-P20 ──
M_raw_h, M_prob_h, chars_h, idx_h = construir_matriz_transicion_intrapalabra(hebrew_words_chars)
words_syl_h, thresh_h = segmentar_tp(hebrew_words_chars, M_prob_h, idx_h, 20)

all_syl_h = [s for w in words_syl_h for s in w]
syl_freq_h = Counter(all_syl_h)
VOCAB_SYL_H = sorted(syl_freq_h.keys(), key=lambda s: syl_freq_h[s], reverse=True)
VOCAB_SYL_H_FUNC = [s for s in VOCAB_SYL_H if syl_freq_h[s] >= 5]

print(f'Hebrew syllables: {len(VOCAB_SYL_H)} types, {len(VOCAB_SYL_H_FUNC)} functional (freq>=5)')
print(f'Mean syllables/word: {np.mean([len(w) for w in words_syl_h]):.2f}')
print(f'Mean chars/syllable: {np.mean([len(s) for s in all_syl_h]):.2f}')
print(f'Threshold: {thresh_h:.4f}')

In [ ]:
# ── Cell 4: LSTM Character-Level LM ──────────────────────────────────

class CharDataset(Dataset):
    def __init__(self, words, char2idx):
        self.samples = []
        bos = char2idx['<w>']
        eos = char2idx['</w>']
        for word in words:
            indices = [bos, bos] + [char2idx.get(c, char2idx.get('<unk>', 0)) for c in word] + [eos]
            for i in range(2, len(indices)):
                ctx = indices[i-2:i]
                target = indices[i]
                self.samples.append((ctx, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ctx, target = self.samples[idx]
        return torch.tensor(ctx, dtype=torch.long), torch.tensor(target, dtype=torch.long)


class NeuralCharLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=192, n_layers=2, dropout=0.2):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers,
                           dropout=dropout if n_layers > 1 else 0.0,
                           batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.char2idx = None
        self.idx2char = None

    def forward(self, x, hidden=None):
        emb = self.dropout(self.embedding(x))
        out, hidden = self.lstm(emb, hidden)
        out = self.dropout(out)
        logits = self.fc(out)
        return logits, hidden

    def batch_log_prob(self, words, temperature=1.0):
        """Score a list of words (each word = list of characters)."""
        self.eval()
        bos = self.char2idx['<w>']
        eos = self.char2idx['</w>']
        unk = self.char2idx.get('<unk>', 0)
        with torch.no_grad():
            all_ctx, all_tgt = [], []
            word_boundaries = [0]
            for word in words:
                seq = [bos, bos] + [self.char2idx.get(c, unk) for c in word] + [eos]
                for i in range(2, len(seq)):
                    all_ctx.append([seq[i-2], seq[i-1]])
                    all_tgt.append(seq[i])
                word_boundaries.append(len(all_ctx))
            n_total = len(all_ctx)
            if n_total == 0:
                return np.zeros(len(words))
            ctx_t = torch.tensor(all_ctx, device=DEVICE)
            tgt_t = torch.tensor(all_tgt, device=DEVICE)
            CHUNK = 16384
            all_lps = []
            for start in range(0, n_total, CHUNK):
                end = min(start + CHUNK, n_total)
                logits, _ = self.forward(ctx_t[start:end])
                logits_last = logits[:, -1, :] / temperature
                log_p = torch.log_softmax(logits_last, dim=-1)
                target_lp = log_p.gather(1, tgt_t[start:end].unsqueeze(1)).squeeze(1)
                all_lps.append(target_lp)
            all_lps = torch.cat(all_lps).cpu().numpy() / math.log(2)
            word_lps = np.zeros(len(words))
            for wi in range(len(words)):
                s, e = word_boundaries[wi], word_boundaries[wi + 1]
                if s < e:
                    word_lps[wi] = all_lps[s:e].sum()
            return word_lps


# Build char vocab
special = ['<pad>', '<w>', '</w>', '<unk>']
char_vocab = special + target_chars_heb
char2idx = {c: i for i, c in enumerate(char_vocab)}
idx2char = {i: c for c, i in char2idx.items()}

# Train/val split
rng_split = np.random.default_rng(42)
n_val = max(1000, len(hebrew_words_chars) // 10)
idx_all = rng_split.permutation(len(hebrew_words_chars))
val_idx = idx_all[:n_val]
train_idx = idx_all[n_val:]
train_words = [hebrew_words_chars[i] for i in train_idx]
val_words = [hebrew_words_chars[i] for i in val_idx]

# Build datasets
train_ds = CharDataset(train_words, char2idx)
val_ds = CharDataset(val_words, char2idx)
train_dl = DataLoader(train_ds, batch_size=512, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=1024, shuffle=False)

# Train LSTM
neural_lm = NeuralCharLM(len(char_vocab), embed_dim=32, hidden_dim=192, n_layers=2, dropout=0.2).to(DEVICE)
neural_lm.char2idx = char2idx
neural_lm.idx2char = idx2char
optimizer = torch.optim.Adam(neural_lm.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print(f'Training LSTM: {len(train_ds)} samples, {len(val_ds)} val samples')
best_val_loss = float('inf')
patience, max_patience = 0, 5

for epoch in range(30):
    neural_lm.train()
    total_loss, n_batches = 0, 0
    for ctx_b, tgt_b in train_dl:
        ctx_b, tgt_b = ctx_b.to(DEVICE), tgt_b.to(DEVICE)
        logits, _ = neural_lm(ctx_b)
        loss = criterion(logits[:, -1, :], tgt_b)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(neural_lm.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    
    neural_lm.eval()
    val_loss, n_val_b = 0, 0
    with torch.no_grad():
        for ctx_b, tgt_b in val_dl:
            ctx_b, tgt_b = ctx_b.to(DEVICE), tgt_b.to(DEVICE)
            logits, _ = neural_lm(ctx_b)
            loss = criterion(logits[:, -1, :], tgt_b)
            val_loss += loss.item()
            n_val_b += 1
    
    avg_val = val_loss / max(n_val_b, 1)
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_model_state = {k: v.cpu().clone() for k, v in neural_lm.state_dict().items()}
        patience = 0
    else:
        patience += 1
    
    if epoch % 5 == 0 or patience > 0:
        print(f'  Epoch {epoch:2d}: train={total_loss/n_batches:.4f}, val={avg_val:.4f}, patience={patience}')
    
    if patience >= max_patience:
        print(f'  Early stopping at epoch {epoch}')
        break

neural_lm.load_state_dict(best_model_state)
neural_lm = neural_lm.to(DEVICE)
print(f'Best val loss: {best_val_loss:.4f}')

In [ ]:
# ── Cell 5: Syllable-to-Syllable Mapping State ───────────────────────

class SylMappingState:
    """Maps Voynich syllable types to Hebrew syllable types.
    Decoding: Voynich word → list of Voynich syllables → list of Hebrew syllables → character sequence.
    """
    def __init__(self, voynich_syls, hebrew_syls):
        self.voynich_syls = list(voynich_syls)  # list of Voynich syllable types
        self.hebrew_syls = list(hebrew_syls)     # list of Hebrew syllable types
        self.mapping = {}  # voynich_syl_tuple → hebrew_syl_tuple

    def set_mapping(self, mapping_dict):
        self.mapping = dict(mapping_dict)

    def propose_change(self, rng):
        new = SylMappingState(self.voynich_syls, self.hebrew_syls)
        new.mapping = dict(self.mapping)
        idx = rng.integers(len(self.voynich_syls))
        syl = self.voynich_syls[idx]
        new.mapping[syl] = self.hebrew_syls[rng.integers(len(self.hebrew_syls))]
        return new, syl

    def decode_word(self, word_syls):
        """Decode a Voynich word (list of syllable tuples) → Hebrew character list."""
        chars = []
        for syl in word_syls:
            heb_syl = self.mapping.get(syl)
            if heb_syl is not None:
                chars.extend(heb_syl)  # Hebrew syllable is a tuple of chars
            else:
                chars.append('?')
        return chars

    def decode_corpus(self, words_as_syls):
        return [self.decode_word(w) for w in words_as_syls]


def init_frequency_syl_mapping(words_syl_v, syl_freq_v, words_syl_h, syl_freq_h,
                               voynich_syls, hebrew_syls):
    """Initialize mapping by matching frequency ranks."""
    v_ranked = sorted(voynich_syls, key=lambda s: syl_freq_v.get(s, 0), reverse=True)
    h_ranked = sorted(hebrew_syls, key=lambda s: syl_freq_h.get(s, 0), reverse=True)
    mapping = {}
    for i, vs in enumerate(v_ranked):
        mapping[vs] = h_ranked[i % len(h_ranked)]
    return mapping


def build_syl_word_index(words_as_syls):
    """Map each syllable type to the word indices containing it."""
    idx = defaultdict(list)
    for wi, word in enumerate(words_as_syls):
        for syl in set(word):
            idx[syl].append(wi)
    return idx


# ── Scoring ──

def score_lm_syl(state, words_syl_sample, neural_lm, temperature=2.0):
    """Score by decoding syllables to characters and using LSTM."""
    decoded = state.decode_corpus(words_syl_sample)
    word_lps = neural_lm.batch_log_prob(decoded, temperature)
    return float(np.mean(word_lps)), word_lps


def score_cardinality_syl(state, K_target=None):
    """Penalize if too few distinct Hebrew syllables used."""
    K = len(set(state.mapping.values()))
    if K_target is None:
        return 0.0
    if K >= K_target:
        return 0.0
    return -((K_target - K) ** 2) * 0.01


def score_entropy_syl(state):
    counts = Counter(state.mapping.values())
    if len(counts) <= 1:
        return 0.0
    total = sum(counts.values())
    probs = np.array([c / total for c in counts.values()])
    probs = probs[probs > 0]
    H = -np.sum(probs * np.log2(probs))
    H_max = np.log2(len(counts))
    return H / H_max if H_max > 1e-10 else 0.0


def score_total_syl(state, words_syl_sample, neural_lm, temperature,
                    w_lm=2.0, w_ent=0.3, w_card=0.3, K_target=None):
    s_lm, word_lps = score_lm_syl(state, words_syl_sample, neural_lm, temperature)
    s_ent = score_entropy_syl(state)
    s_card = score_cardinality_syl(state, K_target)
    total = w_lm * s_lm + w_ent * s_ent + w_card * s_card
    return total, word_lps


def score_incremental_syl(proposal, changed_syl, words_syl_sample, neural_lm,
                          temperature, syl_word_idx, prev_word_lps,
                          w_lm=2.0, w_ent=0.3, w_card=0.3, K_target=None):
    """Only rescore words containing the changed syllable."""
    new_lps = prev_word_lps.copy()
    affected = syl_word_idx.get(changed_syl, [])
    if affected:
        affected_decoded = [proposal.decode_word(words_syl_sample[wi]) for wi in affected]
        affected_scores = neural_lm.batch_log_prob(affected_decoded, temperature)
        for j, wi in enumerate(affected):
            new_lps[wi] = affected_scores[j]
    s_lm = float(np.mean(new_lps))
    s_ent = score_entropy_syl(proposal)
    s_card = score_cardinality_syl(proposal, K_target)
    total = w_lm * s_lm + w_ent * s_ent + w_card * s_card
    return total, new_lps


# Prepare syllable inventories
print(f'\nMapping space: {len(VOCAB_SYL_V_FUNC)} Voynich syls → {len(VOCAB_SYL_H_FUNC)} Hebrew syls')
print(f'Ratio |Σ|/|A|: {len(VOCAB_SYL_V_FUNC)}/{len(VOCAB_SYL_H_FUNC)} = {len(VOCAB_SYL_V_FUNC)/max(len(VOCAB_SYL_H_FUNC),1):.2f}')

In [ ]:
# ── Cell 6: Multi-Seed EM ────────────────────────────────────────────

N_SEEDS = 10
T_VALUES = [2.0, 3.0]
SAMPLE_EM = 3000
EM_ITERS = 1500
EM_PATIENCE = 80
PROPOSALS_PER_ITER = 300
SCORE_KW = dict(w_lm=2.0, w_ent=0.3, w_card=0.3, K_target=None)

# Subsample for EM
rng_em = np.random.default_rng(42)
if len(words_syl_v) > SAMPLE_EM:
    idx_em = rng_em.choice(len(words_syl_v), size=SAMPLE_EM, replace=False)
    words_em = [words_syl_v[i] for i in idx_em]
else:
    words_em = words_syl_v

syl_word_idx_em = build_syl_word_index(words_em)

# Base frequency mapping
base_mapping = init_frequency_syl_mapping(
    words_syl_v, syl_freq_v, words_syl_h, syl_freq_h,
    VOCAB_SYL_V_FUNC, VOCAB_SYL_H_FUNC)

all_em_results = []

for T in T_VALUES:
    print(f'\n=== Temperature T={T} ===')
    for seed_idx in range(N_SEEDS):
        rng = np.random.default_rng(42 + seed_idx * 137 + int(T * 1000))
        
        state = SylMappingState(VOCAB_SYL_V_FUNC, VOCAB_SYL_H_FUNC)
        if seed_idx < 5:
            # Frequency + scramble 1/3
            mapping_init = dict(base_mapping)
            n_scramble = max(1, len(VOCAB_SYL_V_FUNC) // 3)
            scramble_idx = rng.choice(len(VOCAB_SYL_V_FUNC), size=n_scramble, replace=False)
            for si in scramble_idx:
                s = VOCAB_SYL_V_FUNC[si]
                mapping_init[s] = VOCAB_SYL_H_FUNC[rng.integers(len(VOCAB_SYL_H_FUNC))]
            state.set_mapping(mapping_init)
        else:
            # Fully random
            mapping_init = {s: VOCAB_SYL_H_FUNC[rng.integers(len(VOCAB_SYL_H_FUNC))]
                           for s in VOCAB_SYL_V_FUNC}
            state.set_mapping(mapping_init)
        
        # Also map rare syllables to most common Hebrew syllable
        for syl in VOCAB_SYL_V:
            if syl not in state.mapping:
                state.mapping[syl] = VOCAB_SYL_H_FUNC[0]  # fallback
        
        # Initial score
        best_score, best_word_lps = score_total_syl(
            state, words_em, neural_lm, T, **SCORE_KW)
        best_state = state
        no_improve = 0
        
        t0 = time.time()
        for it in range(EM_ITERS):
            improved = False
            for _ in range(PROPOSALS_PER_ITER):
                proposal, changed_syl = best_state.propose_change(rng)
                # Also keep rare syllable fallbacks
                for syl in VOCAB_SYL_V:
                    if syl not in proposal.mapping:
                        proposal.mapping[syl] = VOCAB_SYL_H_FUNC[0]
                
                prop_score, prop_lps = score_incremental_syl(
                    proposal, changed_syl, words_em, neural_lm, T,
                    syl_word_idx_em, best_word_lps, **SCORE_KW)
                if prop_score > best_score:
                    best_state = proposal
                    best_score = prop_score
                    best_word_lps = prop_lps
                    improved = True
            
            if not improved:
                no_improve += 1
                if no_improve >= EM_PATIENCE:
                    break
            else:
                no_improve = 0
        
        elapsed = time.time() - t0
        K_eff = len(set(best_state.mapping.values()))
        
        result = {
            'seed': seed_idx, 'T': T, 'score': best_score,
            'K_eff': K_eff, 'elapsed': elapsed,
            'state': best_state, 'word_lps': best_word_lps,
            'iter_converged': it + 1,
        }
        all_em_results.append(result)
        
        init_type = 'freq+scramble' if seed_idx < 5 else 'random'
        print(f'  Seed {seed_idx:2d} [{init_type:13s}]: score={best_score:.4f}, '
              f'K={K_eff:3d}, iter={it+1:4d}, {elapsed:.0f}s')

# Select best
BEST = max(all_em_results, key=lambda r: r['score'])
print(f'\nBest: T={BEST["T"]}, seed={BEST["seed"]}, score={BEST["score"]:.4f}, K_eff={BEST["K_eff"]}')

In [ ]:
# ── Cell 7: MCMC Multi-Chain ─────────────────────────────────────────

T_mcmc = BEST['T']
N_CHAINS = 4
N_ITER = 15000
BURN_IN = 3000
THIN = 5

print(f'MCMC: {N_CHAINS} chains, {N_ITER} iter, T={T_mcmc}')

mcmc_chains = []
for chain_id in range(N_CHAINS):
    rng = np.random.default_rng(42 + chain_id * 999)
    
    # Initialize from best EM + small perturbation
    state = SylMappingState(VOCAB_SYL_V_FUNC, VOCAB_SYL_H_FUNC)
    init_map = dict(BEST['state'].mapping)
    # Perturb ~10% of mappings
    n_perturb = max(1, len(VOCAB_SYL_V_FUNC) // 10)
    perturb_idx = rng.choice(len(VOCAB_SYL_V_FUNC), size=n_perturb, replace=False)
    for pi in perturb_idx:
        s = VOCAB_SYL_V_FUNC[pi]
        init_map[s] = VOCAB_SYL_H_FUNC[rng.integers(len(VOCAB_SYL_H_FUNC))]
    state.set_mapping(init_map)
    # Ensure rare syllables covered
    for syl in VOCAB_SYL_V:
        if syl not in state.mapping:
            state.mapping[syl] = VOCAB_SYL_H_FUNC[0]
    
    cur_score, cur_lps = score_total_syl(state, words_em, neural_lm, T_mcmc, **SCORE_KW)
    
    chain_mappings = []
    chain_scores = []
    n_accepted = 0
    
    # Adaptive temperature
    temp_scale = 1.0
    
    t0 = time.time()
    for it in range(N_ITER):
        proposal, changed_syl = state.propose_change(rng)
        for syl in VOCAB_SYL_V:
            if syl not in proposal.mapping:
                proposal.mapping[syl] = VOCAB_SYL_H_FUNC[0]
        
        prop_score, prop_lps = score_incremental_syl(
            proposal, changed_syl, words_em, neural_lm, T_mcmc,
            syl_word_idx_em, cur_lps, **SCORE_KW)
        
        delta = prop_score - cur_score
        if delta > 0 or rng.random() < math.exp(delta * temp_scale):
            state = proposal
            cur_score = prop_score
            cur_lps = prop_lps
            n_accepted += 1
        
        if it % THIN == 0:
            chain_mappings.append(dict(state.mapping))
            chain_scores.append(cur_score)
        
        # Adaptive temperature every 500 steps
        if it > 0 and it % 500 == 0:
            accept_rate = n_accepted / (it + 1)
            if accept_rate < 0.3:
                temp_scale *= 0.9
            elif accept_rate > 0.7:
                temp_scale *= 1.1
    
    elapsed = time.time() - t0
    accept_rate = n_accepted / N_ITER
    mcmc_chains.append({
        'chain_mappings': chain_mappings,
        'chain_scores': chain_scores,
        'accept_rate': accept_rate,
    })
    print(f'  Chain {chain_id}: accept={accept_rate:.3f}, final_score={cur_score:.4f}, {elapsed:.0f}s')

# Global best from MCMC
global_best_map = None
global_best_score = -float('inf')
for ch in mcmc_chains:
    for m, s in zip(ch['chain_mappings'], ch['chain_scores']):
        if s > global_best_score:
            global_best_score = s
            global_best_map = m

final_state = SylMappingState(VOCAB_SYL_V_FUNC, VOCAB_SYL_H_FUNC)
final_state.set_mapping(global_best_map)
for syl in VOCAB_SYL_V:
    if syl not in final_state.mapping:
        final_state.mapping[syl] = VOCAB_SYL_H_FUNC[0]

print(f'\nGlobal best MCMC score: {global_best_score:.4f}')
print(f'K_eff: {len(set(final_state.mapping.values()))}')

In [ ]:
# ── Cell 8: 5-Test Scorecard ─────────────────────────────────────────

mapping_final = final_state.mapping

# ── Posterior + R-hat ──
burn_idx = max(BURN_IN // THIN, 1)
all_post_burn = []
for ch in mcmc_chains:
    mappings = ch['chain_mappings']
    if len(mappings) > burn_idx:
        all_post_burn.extend(mappings[burn_idx:])

# R-hat per Voynich syllable
rhat = {}
for syl in VOCAB_SYL_V_FUNC:
    # Most common target for this syllable
    target_counts = Counter(m.get(syl) for m in all_post_burn if syl in m)
    if not target_counts:
        rhat[syl] = float('nan')
        continue
    top_target = max(target_counts, key=target_counts.get)
    chain_means, chain_ns = [], []
    for ch in mcmc_chains:
        post_burn = ch['chain_mappings'][burn_idx:] if len(ch['chain_mappings']) > burn_idx else []
        if not post_burn:
            continue
        indicators = [1.0 if m.get(syl) == top_target else 0.0 for m in post_burn]
        chain_means.append(np.mean(indicators))
        chain_ns.append(len(indicators))
    if len(chain_means) < 2:
        rhat[syl] = float('nan')
        continue
    m_ch = len(chain_means)
    n_per = np.mean(chain_ns)
    grand_mean = np.mean(chain_means)
    B = n_per / (m_ch - 1) * sum((cm - grand_mean) ** 2 for cm in chain_means)
    W_terms = []
    for ch in mcmc_chains:
        post_burn = ch['chain_mappings'][burn_idx:] if len(ch['chain_mappings']) > burn_idx else []
        if not post_burn:
            continue
        indicators = [1.0 if m.get(syl) == top_target else 0.0 for m in post_burn]
        W_terms.append(np.var(indicators, ddof=1) if len(indicators) > 1 else 0.0)
    W = np.mean(W_terms)
    if W < 1e-12:
        rhat[syl] = 1.0 if B < 1e-12 else float('inf')
    else:
        var_hat = (1 - 1 / n_per) * W + (1 / n_per) * B
        rhat[syl] = math.sqrt(var_hat / W)

rhat_vals = [v for v in rhat.values() if isinstance(v, (int, float)) and not np.isnan(v) and not np.isinf(v)]
pct_converged = 100 * sum(1 for v in rhat_vals if v < 1.1) / max(len(rhat_vals), 1)
print(f'R-hat < 1.1: {pct_converged:.1f}% of {len(rhat_vals)} syllables')

# ── Test 1: Collapse coherence (via positional JSD) ──
# Group Voynich syllables that map to same Hebrew syllable
def syl_positional_dist(syl, words_syl):
    counts = np.array([0.0, 0.0, 0.0])  # onset, middle, coda
    for word in words_syl:
        for i, s in enumerate(word):
            if s == syl:
                if len(word) == 1:
                    counts[0] += 0.5; counts[2] += 0.5
                elif i == 0:
                    counts[0] += 1
                elif i == len(word) - 1:
                    counts[2] += 1
                else:
                    counts[1] += 1
    total = counts.sum()
    return counts / total if total > 0 else np.array([1/3, 1/3, 1/3])

pos_dists_syl = {s: syl_positional_dist(s, words_syl_v) for s in VOCAB_SYL_V_FUNC}

groups = defaultdict(list)
for vs, hs in mapping_final.items():
    if vs in VOCAB_SYL_V_FUNC:
        groups[hs].append(vs)

z_scores = []
for hs, v_syls in groups.items():
    if len(v_syls) < 3:
        continue
    pairs = list(combinations(v_syls, 2))
    real_jsds = [jensenshannon(pos_dists_syl[a] + 1e-10, pos_dists_syl[b] + 1e-10) for a, b in pairs]
    real_mean = np.mean(real_jsds)
    null_means = []
    rng_null = np.random.default_rng(42)
    for _ in range(200):
        random_syls = [VOCAB_SYL_V_FUNC[rng_null.integers(len(VOCAB_SYL_V_FUNC))] for _ in v_syls]
        random_pairs = list(combinations(random_syls, 2))
        if random_pairs:
            null_means.append(np.mean([jensenshannon(pos_dists_syl[a] + 1e-10, pos_dists_syl[b] + 1e-10)
                                       for a, b in random_pairs]))
    if null_means:
        null_mu, null_std = np.mean(null_means), np.std(null_means)
        z = (real_mean - null_mu) / max(null_std, 1e-10)
        z_scores.append(z)

mean_z = np.mean(z_scores) if z_scores else 0.0
print(f'\nTest 1 - Collapse z-score: {mean_z:.3f}')
v1 = 'PASS' if mean_z > -1.0 else ('MARGINAL' if mean_z > -1.5 else 'FAIL')
print(f'  Verdict: {v1}')

# ── Test 2: N-gram percentilee ──
decoded_corpus = final_state.decode_corpus(words_syl_v)

# Build Hebrew bigram/trigram reference
def count_ngrams(words, n=2):
    counts = Counter()
    for w in words:
        seq = ['<w>'] + list(w) + ['</w>']
        for i in range(len(seq) - n + 1):
            counts[tuple(seq[i:i+n])] += 1
    return counts

ref_bi = set(count_ngrams(hebrew_words_chars, 2).keys())
ref_tri = set(count_ngrams(hebrew_words_chars, 3).keys())

def ngram_fraction(decoded_words, ref_bi, ref_tri):
    total_bi, match_bi = 0, 0
    total_tri, match_tri = 0, 0
    for w in decoded_words:
        seq = ['<w>'] + list(w) + ['</w>']
        for i in range(len(seq) - 1):
            total_bi += 1
            if tuple(seq[i:i+2]) in ref_bi:
                match_bi += 1
        for i in range(len(seq) - 2):
            total_tri += 1
            if tuple(seq[i:i+3]) in ref_tri:
                match_tri += 1
    frac_bi = match_bi / max(total_bi, 1)
    frac_tri = match_tri / max(total_tri, 1)
    return (frac_bi + frac_tri) / 2

real_ngram = ngram_fraction(decoded_corpus, ref_bi, ref_tri)

# Null distribution: 1000 random syl-to-syl mappings
rng_null = np.random.default_rng(42)
null_ngrams = []
for _ in range(1000):
    null_map = {s: VOCAB_SYL_H_FUNC[rng_null.integers(len(VOCAB_SYL_H_FUNC))]
                for s in VOCAB_SYL_V}
    null_state = SylMappingState(VOCAB_SYL_V_FUNC, VOCAB_SYL_H_FUNC)
    null_state.set_mapping(null_map)
    null_decoded = null_state.decode_corpus(words_syl_v)
    null_ngrams.append(ngram_fraction(null_decoded, ref_bi, ref_tri))

ngram_pct = 100 * np.mean([real_ngram > x for x in null_ngrams])  # R1 fix
print(f'\nTest 2 - N-gram: real={real_ngram:.4f}, percentile={ngram_pct:.1f}%')
v2 = 'PASS' if ngram_pct > 15 else ('MARGINAL' if ngram_pct > 5 else 'FAIL')
print(f'  Verdict: {v2}')

# ── Test 3: Root matching ──
HEBREW_ROOTS = [
    'כתב', 'דבר', 'שמר', 'עשה', 'נתן', 'הלך', 'אמר', 'ידע', 'בוא', 'ראה',
    'שמע', 'לקח', 'עבד', 'מלך', 'ברך', 'קדש', 'זכר', 'שלח', 'ישב', 'עמד',
    'גדל', 'חיה', 'מות', 'אכל', 'שתה', 'לבש', 'רחץ', 'חכם', 'טהר', 'צדק',
    'שפט', 'משל', 'ספר', 'למד', 'בנה', 'פתח', 'סגר', 'קרא', 'נשא', 'שוב',
]

def count_roots(decoded_words, root_list):
    found = set()
    n_matches = 0
    for word in decoded_words:
        word_str = ''.join(word)
        for root in root_list:
            # Check if all 3 consonants appear in order
            pos = 0
            matched = True
            for ch in root:
                idx = word_str.find(ch, pos)
                if idx == -1:
                    matched = False
                    break
                pos = idx + 1
            if matched:
                found.add(root)
                n_matches += 1
    return found, n_matches

roots_found, n_root_matches = count_roots(decoded_corpus, HEBREW_ROOTS)
print(f'\nTest 3 - Roots: {len(roots_found)} unique roots, {n_root_matches} total matches')
if roots_found:
    print(f'  Found: {sorted(roots_found)}')
v3 = 'PASS' if len(roots_found) >= 10 else ('MARGINAL' if len(roots_found) >= 5 else 'FAIL')
print(f'  Verdict: {v3}')

# ── Test 4: Cardinality ──
K_eff = len(set(final_state.mapping.values()))
K_target = len(VOCAB_SYL_H_FUNC)
card_ratio = K_eff / K_target
print(f'\nTest 4 - Cardinality: K_eff={K_eff}, K_target={K_target}, ratio={card_ratio:.3f}')
v4 = 'PASS' if card_ratio > 0.6 else ('MARGINAL' if card_ratio > 0.4 else 'FAIL')
print(f'  Verdict: {v4}')

# ── Test 5: Word length distribution ──
decoded_lengths = [len(w) for w in decoded_corpus]
hebrew_lengths = [len(w) for w in hebrew_words_chars]
mean_dec = np.mean(decoded_lengths)
mean_heb = np.mean(hebrew_lengths)
std_heb = np.std(hebrew_lengths)
z_len = abs(mean_dec - mean_heb) / max(std_heb, 1e-10)
print(f'\nTest 5 - Word length: decoded={mean_dec:.2f}, Hebrew={mean_heb:.2f}, z={z_len:.2f}')
v5 = 'PASS' if z_len < 2.0 else ('MARGINAL' if z_len < 3.0 else 'FAIL')
print(f'  Verdict: {v5}')

# ── Summary ──
verdicts = [v1, v2, v3, v4, v5]
n_pass = sum(1 for v in verdicts if v == 'PASS')
n_marg = sum(1 for v in verdicts if v == 'MARGINAL')
n_fail = sum(1 for v in verdicts if v == 'FAIL')
print(f'\n{"="*60}')
print(f'SCORECARD: {n_pass}+/{n_marg}±/{n_fail}−')
print(f'R-hat < 1.1: {pct_converged:.1f}%')
print(f'N-gram percentile: {ngram_pct:.1f}%')
print(f'Roots: {len(roots_found)}')
print(f'K_eff: {K_eff} / {K_target}')
print(f'{"="*60}')

In [ ]:
# ── Cell 9: Decoded text samples + analysis ──────────────────────────

print('=== Sample decoded words ===')
for i in range(min(30, len(words_syl_v))):
    voynich_word = palabras[i]
    voynich_syls = words_syl_v[i]
    decoded = final_state.decode_word(voynich_syls)
    v_str = '-'.join(''.join(s) for s in voynich_syls)
    d_str = ''.join(decoded)
    print(f'  {" ".join(voynich_word):20s} → [{v_str:20s}] → {d_str}')

# Mapping statistics
print(f'\n=== Mapping statistics ===')
print(f'Voynich syllables mapped: {len(mapping_final)}')
print(f'Hebrew syllables used: {K_eff}')
print(f'Fan-out (V→H): mean={np.mean([len([v for v,h in mapping_final.items() if h==hs]) for hs in set(mapping_final.values())]):.1f}')

# Show top mappings by frequency
print(f'\n=== Top 20 mappings (by Voynich frequency) ===')
for syl in VOCAB_SYL_V_FUNC[:20]:
    heb_syl = mapping_final.get(syl, '?')
    heb_str = ''.join(heb_syl) if isinstance(heb_syl, tuple) else str(heb_syl)
    v_str = ''.join(syl)
    print(f'  {v_str:12s} (freq={syl_freq_v[syl]:5d}) → {heb_str}')

In [ ]:
# ── Cell 10: Export results ──────────────────────────────────────────

results = {
    'metadata': {
        'model': 'Syllable-to-Syllable (Model 12)',
        'description': 'Maps Voynich syllable types to Hebrew syllable types, scored by LSTM CharLM',
        'voynich_syllables': len(VOCAB_SYL_V),
        'voynich_syllables_functional': len(VOCAB_SYL_V_FUNC),
        'hebrew_syllables': len(VOCAB_SYL_H),
        'hebrew_syllables_functional': len(VOCAB_SYL_H_FUNC),
        'ratio_sigma_A': round(len(VOCAB_SYL_V_FUNC) / max(len(VOCAB_SYL_H_FUNC), 1), 2),
        'segmentation': 'TP-P20',
        'lm': 'LSTM 2-layer (embed=32, hidden=192)',
        'temperature': T_mcmc,
        'n_seeds': N_SEEDS * len(T_VALUES),
        'mcmc_chains': N_CHAINS,
        'mcmc_iter': N_ITER,
    },
    'scorecard': {
        'collapse_z': round(mean_z, 3),
        'ngram_percentile': round(ngram_pct, 1),
        'roots_found': len(roots_found),
        'roots_list': sorted(roots_found),
        'root_matches_total': n_root_matches,
        'cardinality_ratio': round(card_ratio, 3),
        'K_eff': K_eff,
        'word_length_z': round(z_len, 2),
        'rhat_pct': round(pct_converged, 1),
        'verdicts': verdicts,
        'summary': f'{n_pass}+/{n_marg}±/{n_fail}−',
    },
    'best_em': {
        'score': round(BEST['score'], 4),
        'temperature': BEST['T'],
        'seed': BEST['seed'],
        'K_eff': BEST['K_eff'],
    },
    'mcmc_best_score': round(global_best_score, 4),
    'val_loss': round(best_val_loss, 4),
}

with open('syl2syl_shuf_resultados.json', 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print('Results saved to syl2syl_results.json')
print(json.dumps(results, indent=2, ensure_ascii=False))
# ── R1: store decoding input + mapping for the unified n-gram re-evaluation ──
_j = lambda u: list(u) if isinstance(u, (tuple, list)) else [u]
with open('syl2syl_shuf_mapping.json', 'w') as f:
    json.dump({'units': 'syllables' if True else 'glyphs',
               'words': [[_j(k) for k in w] for w in words_syl_v],
               'mapping': [[_j(k), _j(v)] for k, v in mapping_final.items()]},
              f, ensure_ascii=False)
print('Mapping saved to syl2syl_shuf_mapping.json')
